# nanoDiT-JAX on Colab

Train a 129.6M-parameter, class-conditional flow model on butterflies. JAX port of [nanoDiT](https://github.com/sayakpaul/nanoDiT).

The [nanoDiT-JAX repository](https://github.com/sayakpaul/nanoDiT-JAX) also provides instructions for running with the [Colab CLI](https://github.com/sayakpaul/nanoDiT-JAX#colab-cli) and on [Google Cloud (GCP)](https://github.com/sayakpaul/nanoDiT-JAX#google-cloud).

Select **Runtime → Change runtime type → TPU v6e-1**, then run the cells in order. The demo trains for 10 epochs; early samples will be rough. [Colab storage is temporary](https://research.google.com/colaboratory/faq.html): download outputs before ending the runtime.

In [ ]:
#@title 1. Install
from pathlib import Path
import json
import subprocess
import sys
import tarfile

ROOT = Path("/content/nanoDiT-JAX")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/sayakpaul/nanoDiT-JAX.git", str(ROOT)], check=True)
PYTHON = ROOT / ".venv/bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(ROOT / ".venv")], check=True)
subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON), "install",
                "-q", "-r", str(ROOT / "requirements.txt")], check=True)
OUTPUT = ROOT / "runs/colab"
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())

In [ ]:
#@title 2. Check the TPU
# Keep JAX in subprocesses so the notebook kernel does not reserve the TPU.
subprocess.run([str(PYTHON), "-c", """
import jax
print('JAX:', jax.__version__, 'Devices:', jax.devices(), flush=True)
assert jax.default_backend() == 'tpu', 'Select a TPU runtime first.'
assert jax.device_count() == 1, 'This notebook requires one TPU device (v6e-1).'
assert 'v6' in jax.devices()[0].device_kind.lower(), 'Select a v6e-1 TPU.'
"""], check=True)

In [ ]:
#@title Optional: restore a downloaded archive
# Upload the archive through Colab's Files pane, then enter its /content/... path.
restore_archive = "" #@param {type:"string"}
if restore_archive:
    if (OUTPUT / "latest.msgpack").exists():
        raise FileExistsError("A checkpoint already exists; use a fresh runtime to restore another archive.")
    with tarfile.open(restore_archive) as archive:
        archive.extractall(ROOT, filter="data")
    print("Restored:", OUTPUT)

The **demo** saves live-weight and EMA samples every 5 epochs. **Full** uses the original 600-epoch recipe, sampling every 10 epochs and checkpointing every 25. Both use batch 128, microbatch 8, bf16, AdamW at `1e-4`, and EMA warmup capped at `0.999`.

An existing checkpoint resumes automatically. Switch from demo to full to continue training.

In [ ]:
#@title 3. Train
run_mode = "demo" #@param ["demo", "full"]
epochs = 600 if run_mode == "full" else 10
sample_every = 500 if run_mode == "full" else 250
checkpoint_every = 1250 if run_mode == "full" else 250
command = [str(PYTHON), "-u", "train.py", "--require-tpu", "--download",
           "--epochs", str(epochs), "--batch-size", "128", "--microbatch-size", "8", "--no-remat",
           "--sample-every", str(sample_every), "--checkpoint-every", str(checkpoint_every),
           "--sample-ema", "--ema-decay", "0.999", "--output-dir", str(OUTPUT)]
checkpoint = OUTPUT / "latest.msgpack"
if checkpoint.exists():
    command += ["--resume", str(checkpoint)]
status_path = OUTPUT / "status.json"
status = json.loads(status_path.read_text()) if status_path.exists() else {}
if checkpoint.exists() and status.get("status") == "complete" and status.get("epoch", 0) >= epochs:
    print("This run is complete. Preview/download below, or switch demo to full.")
else:
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                process.wait()
            raise

In [ ]:
#@title 4. View the loss and latest samples
from IPython.display import Image, display

plot = OUTPUT / "loss.png"
if plot.exists():
    display(Image(filename=str(plot), width=720))
for pattern, label in [("samples_[0-9]*.png", "Live weights"), ("samples_ema_*.png", "EMA")]:
    samples = sorted(OUTPUT.glob(pattern))
    if samples:
        latest = samples[-1]
        print(f"{label} — step {int(latest.stem.rsplit('_', 1)[-1])}")
        display(Image(filename=str(latest)))
    else:
        print(f"{label}: no samples saved yet.")

Download plots, all intermediate samples and logs. Enable **include_checkpoint** to include the ~2 GB optimizer/RNG checkpoint needed to resume in a new runtime. Restore that archive with the optional cell above before training again.

In [ ]:
#@title 5. Download results
from google.colab import files

include_checkpoint = False #@param {type:"boolean"}
if not OUTPUT.exists():
    raise FileNotFoundError("Run training first.")
archive_path = Path("/content/nanodit-results.tar.gz")
allowed = {".json", ".jsonl", ".png", ".svg"}
if include_checkpoint:
    if not (OUTPUT / "latest.msgpack").exists():
        raise FileNotFoundError("No checkpoint saved yet; continue training first.")
    allowed.add(".msgpack")
with tarfile.open(archive_path, "w:gz") as archive:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file() and path.suffix in allowed:
            archive.add(path, arcname=str(Path("runs/colab") / path.name))
files.download(str(archive_path))